# Real-Data Validation for Preprocessing Input Modes

This notebook validates the updated preprocessing stage on the real dataset configured in `config/paths.yaml`.

It runs the full recommendation pipeline twice:

1. `preprocessing_from_dataframe.yaml`: preprocessing receives a dataframe artifact from the pipeline.
2. `preprocessing_from_path.yaml`: preprocessing loads the CSV itself from config.

Both runs use:

- `save_artifacts=False`
- `return_recommendations=True`

The notebook then validates that both runs produce the same preprocessed dataframe, the same `idx2item` mapping, and the same returned recommendations.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import tempfile
from pathlib import Path
from textwrap import dedent

import pandas as pd

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pipeline import Pipeline

In [ ]:
def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(text).lstrip(), encoding="utf-8")


def digest_mapping(mapping: dict) -> str:
    payload = json.dumps(mapping, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def digest_dataframe(df: pd.DataFrame) -> str:
    normalized = df.reset_index(drop=True)
    payload = normalized.to_json(orient="split", date_format="iso", default_handler=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def build_validation_configs(temp_root: Path) -> dict[str, Path]:
    temp_config_dir = temp_root / "config"
    temp_config_dir.mkdir(parents=True, exist_ok=True)

    repo_config_dir = PROJECT_ROOT / "config"
    paths_yaml = (repo_config_dir / "paths.yaml").resolve()
    preprocessing_from_dataframe = (repo_config_dir / "data_processing" / "preprocessing_from_dataframe.yaml").resolve()
    preprocessing_from_path = (repo_config_dir / "data_processing" / "preprocessing_from_path.yaml").resolve()
    embeddings_yaml = (repo_config_dir / "embeddings" / "function1.yaml").resolve()
    clustering_yaml = (repo_config_dir / "clustering" / "merge_kmeans.yaml").resolve()
    similarity_yaml = (repo_config_dir / "models" / "similarity_matrix" / "from_clusters.yaml").resolve()
    katz_yaml = (repo_config_dir / "models" / "similarity_matrix" / "katz.yaml").resolve()

    dataframe_pipeline_yaml = temp_config_dir / "pipeline_from_dataframe.yaml"
    write_text(
        dataframe_pipeline_yaml,
        f"""
        data:
          csv_path: ${{paths.data_csv}}

        output:
          dir: ${{paths.output_dir}}

        save_intermediates: true

        stage1:
          config_path: {preprocessing_from_dataframe}

        stage2:
          config_path: {embeddings_yaml}

        stage3:
          config_path: {clustering_yaml}

        stage4:
          config_path: {similarity_yaml}

        stage5:
          config_path: {katz_yaml}
        """,
    )

    path_pipeline_yaml = temp_config_dir / "pipeline_from_path.yaml"
    write_text(
        path_pipeline_yaml,
        f"""
        data:
          csv_path: ${{paths.data_csv}}

        output:
          dir: ${{paths.output_dir}}

        save_intermediates: true

        stage1:
          config_path: {preprocessing_from_path}

        stage2:
          config_path: {embeddings_yaml}

        stage3:
          config_path: {clustering_yaml}

        stage4:
          config_path: {similarity_yaml}

        stage5:
          config_path: {katz_yaml}
        """,
    )

    dataframe_base_yaml = temp_config_dir / "base_from_dataframe.yaml"
    write_text(
        dataframe_base_yaml,
        f"""
        project:
          name: classic_methods_validation
          seed: 42

        paths:
          config_path: {paths_yaml}

        pipeline:
          config_path: {dataframe_pipeline_yaml}
        """,
    )

    path_base_yaml = temp_config_dir / "base_from_path.yaml"
    write_text(
        path_base_yaml,
        f"""
        project:
          name: classic_methods_validation
          seed: 42

        paths:
          config_path: {paths_yaml}

        pipeline:
          config_path: {path_pipeline_yaml}
        """,
    )

    return {
        "dataframe_base": dataframe_base_yaml,
        "path_base": path_base_yaml,
    }

In [ ]:
real_pipeline = Pipeline(PROJECT_ROOT / "config" / "base.yaml")
real_data_path = real_pipeline.csv_path
print(f"Real data path: {real_data_path}")
print(f"Data exists: {real_data_path.exists()}")

raw_df = real_pipeline._read_dataframe(real_data_path)
print(f"Raw dataframe shape: {raw_df.shape}")
raw_df.head()

In [ ]:
recommendation_k = 10
temp_dir = tempfile.TemporaryDirectory()
validation_configs = build_validation_configs(Path(temp_dir.name))

pipeline_from_dataframe = Pipeline(validation_configs["dataframe_base"])
artifacts_from_dataframe = pipeline_from_dataframe.run(
    initial_artifacts={"dataframe": raw_df.copy()},
    save_artifacts=False,
    return_recommendations=True,
    recommendation_k=recommendation_k,
)

pipeline_from_path = Pipeline(validation_configs["path_base"])
artifacts_from_path = pipeline_from_path.run(
    save_artifacts=False,
    return_recommendations=True,
    recommendation_k=recommendation_k,
)

processed_df_from_dataframe = artifacts_from_dataframe["dataframe"].value.reset_index(drop=True)
processed_df_from_path = artifacts_from_path["dataframe"].value.reset_index(drop=True)
idx2item_from_dataframe = artifacts_from_dataframe["idx2item"].value
idx2item_from_path = artifacts_from_path["idx2item"].value
recommendations_from_dataframe = artifacts_from_dataframe["recommendations"].value
recommendations_from_path = artifacts_from_path["recommendations"].value

summary = pd.DataFrame(
    [
        {
            "pipeline_variant": "dataframe artifact",
            "preprocessed_rows": len(processed_df_from_dataframe),
            "preprocessed_customers": processed_df_from_dataframe["CustomerID"].nunique(),
            "preprocessed_items": processed_df_from_dataframe["Item Code"].nunique(),
            "recommendation_users": len(recommendations_from_dataframe),
            "output_dir_created": pipeline_from_dataframe.output_dir.exists(),
        },
        {
            "pipeline_variant": "csv path in preprocessing",
            "preprocessed_rows": len(processed_df_from_path),
            "preprocessed_customers": processed_df_from_path["CustomerID"].nunique(),
            "preprocessed_items": processed_df_from_path["Item Code"].nunique(),
            "recommendation_users": len(recommendations_from_path),
            "output_dir_created": pipeline_from_path.output_dir.exists(),
        },
    ]
)

summary

In [ ]:
assert not pipeline_from_dataframe.output_dir.exists(), "save_artifacts=False should not create an output directory for the dataframe-input run."
assert not pipeline_from_path.output_dir.exists(), "save_artifacts=False should not create an output directory for the path-input run."

assert processed_df_from_dataframe.equals(processed_df_from_path), "Preprocessed dataframes differ between the two pipeline modes."
assert idx2item_from_dataframe == idx2item_from_path, "idx2item mappings differ between the two pipeline modes."
assert recommendations_from_dataframe == recommendations_from_path, "Returned recommendations differ between the two pipeline modes."

validation_report = {
    "status": "passed",
    "data_path": str(real_data_path),
    "preprocessed_rows": len(processed_df_from_dataframe),
    "preprocessed_customers": int(processed_df_from_dataframe["CustomerID"].nunique()),
    "preprocessed_items": int(processed_df_from_dataframe["Item Code"].nunique()),
    "recommendation_users": len(recommendations_from_dataframe),
    "recommendation_k": recommendation_k,
    "dataframe_digest": digest_dataframe(processed_df_from_dataframe),
    "idx2item_digest": digest_mapping(idx2item_from_dataframe),
    "recommendations_digest": digest_mapping(recommendations_from_dataframe),
    "validated_behaviors": [
        "The pipeline runs on the real CSV configured in config/paths.yaml.",
        "Preprocessing works when a dataframe artifact is passed into the pipeline.",
        "Preprocessing works when it loads the CSV from config itself.",
        "save_artifacts=False prevents output directory creation.",
        "return_recommendations=True returns ranked predictions for all users.",
        "Both pipeline modes produce identical preprocessing outputs and identical recommendations.",
    ],
}

print(json.dumps(validation_report, indent=2))

In [ ]:
sample_users = list(recommendations_from_dataframe.keys())[:5]
pd.DataFrame(
    {
        "CustomerID": sample_users,
        f"top_{recommendation_k}_item_idx": [recommendations_from_dataframe[user_id] for user_id in sample_users],
    }
)